# LangGraph Fundamentals

In this notebook we'll cover the core building blocks of LangGraph:

1. **Graph Basics** — States, Nodes, Edges, Conditional Routing
2. **Messages & Reducers** — Conversation state, the `add_messages` reducer, `MessagesState`
3. **LLM Workflow Patterns** — Chaining, Parallel execution, Routing
4. **Tool Integration** — `bind_tools`, `ToolNode`, `tools_condition`, agentic loops

Official docs: https://docs.langchain.com/oss/python/langgraph/

In [ ]:
%%capture --no-stderr
%pip install --quiet -U langgraph langchain-openai langchain-ollama langchain-tavily langchain-core pydantic

In [ ]:
import os, getpass

def _set_env(var: str):
    if not os.environ.get(var):
        os.environ[var] = getpass.getpass(f"{var}: ")

_set_env("OPENAI_API_KEY")

---
## 1. Graph Basics (No LLM)

LangGraph has 4 core concepts:

- **State** — A data structure (TypedDict) that gets updated as we execute the graph
- **Nodes** — Python functions that read and update the state
- **Edges** — Connect nodes together (direct `a→b` or conditional `a→b or a→c`)
- **Graph** — The compiled [DAG](https://en.wikipedia.org/wiki/Directed_acyclic_graph) that ties it all together

### A Simple Graph: One Node

Let's start with the simplest possible graph: one state, one node, START → node → END.

In [ ]:
from typing_extensions import TypedDict
from langgraph.graph import StateGraph, START, END
from IPython.display import Image, display

# 1. Define the state
class State(TypedDict):
    state_before_node1: str
    state_after_node1: str

# 2. Define a node (just a function that takes state and returns updated state)
def node1(state):
    print("Passing by node 1")
    state["state_after_node1"] = "Passed by node1"
    return state

# 3. Build the graph
builder = StateGraph(State)
builder.add_node("node1", node1)
builder.add_edge(START, "node1")
builder.add_edge("node1", END)

graph = builder.compile()
display(Image(graph.get_graph().draw_mermaid_png()))

In [ ]:
graph.invoke({"state_before_node1": "Hello from the start!"})

### Conditional Routing

We can route between different nodes based on state using `add_conditional_edges()`. The routing function returns the name of the next node.

<img src="./assets-resources/2025-02-10-12-16-57.png" width=50%>

In [ ]:
from typing import Literal

class State(TypedDict):
    graph_state: str

def node1(state):
    print("Passing by node 1")
    return state

def node2(state):
    print("Passing by node 2")
    state["graph_state"] = "node 2"
    return state

def node3(state):
    print("Passing by node 3")
    state["graph_state"] = "node 3"
    return state

# Routing function — returns the name of the next node
def decision_node(state) -> Literal["node2", "node3"]:
    if state["graph_state"] == "2":
        return "node2"
    else:
        return "node3"

builder = StateGraph(State)
builder.add_node("node1", node1)
builder.add_node("node2", node2)
builder.add_node("node3", node3)

builder.add_edge(START, "node1")
builder.add_conditional_edges("node1", decision_node)
builder.add_edge("node2", END)
builder.add_edge("node3", END)

graph = builder.compile()
display(Image(graph.get_graph().draw_mermaid_png()))

In [ ]:
# Routes to node2
graph.invoke({"graph_state": "2"})

In [ ]:
# Routes to node3
graph.invoke({"graph_state": "3"})

---
## 2. Messages & Reducers

In LLM applications, we typically want to maintain a **conversation history** as messages. LangGraph provides special support for this through **reducers** — functions that control how state keys get updated.

### Message Types

LangChain provides standard message types that LangGraph understands:

In [ ]:
from langchain_core.messages import HumanMessage, AIMessage, SystemMessage, ToolMessage, AnyMessage
from pprint import pprint

messages = [
    SystemMessage(content="You are a helpful assistant."),
    HumanMessage(content="What is LangGraph?"),
    AIMessage(content="LangGraph is a framework for building stateful agents."),
]

for msg in messages:
    print(f"{msg.__class__.__name__}: {msg.content}")

### The Override Problem

By default in LangGraph, returning a new value for a state key **overrides** the previous value. This means messages get lost!

In [ ]:
class MessagesStateNaive(TypedDict):
    messages: list[AnyMessage]  # No reducer — default is override!

def my_node(state):
    new_message = HumanMessage(content="message 1", name="Lucas")
    return {"messages": [new_message]}

def my_node2(state):
    new_message = HumanMessage(content="message 2", name="Lucas")
    return {"messages": [new_message]}

workflow = StateGraph(MessagesStateNaive)
workflow.add_node("test_node", my_node)
workflow.add_node("test_node2", my_node2)
workflow.add_edge(START, "test_node")
workflow.add_edge("test_node", "test_node2")
workflow.add_edge("test_node2", END)
graph = workflow.compile()

# Only message 2 survives — message 1 was overridden!
result = graph.invoke({"messages": []})
print(f"Messages in final state: {len(result['messages'])}")
result

### Reducers: `Annotated` + `add_messages`

To **accumulate** messages instead of overriding, we use a **reducer function**. The `add_messages` reducer appends new messages to the existing list.

```python
messages: Annotated[list[AnyMessage], add_messages]  # ← reducer attached!
```

In [ ]:
from typing import Annotated
from langgraph.graph.message import add_messages

class MessagesStateWithReducer(TypedDict):
    messages: Annotated[list[AnyMessage], add_messages]  # Now messages accumulate!

def my_node(state):
    new_message = HumanMessage(content="message 1", name="Lucas")
    return {"messages": [new_message]}

def my_node2(state):
    new_message = HumanMessage(content="message 2", name="Lucas")
    return {"messages": [new_message]}

workflow = StateGraph(MessagesStateWithReducer)
workflow.add_node("test_node", my_node)
workflow.add_node("test_node2", my_node2)
workflow.add_edge(START, "test_node")
workflow.add_edge("test_node", "test_node2")
workflow.add_edge("test_node2", END)
graph = workflow.compile()

# Both messages are preserved!
result = graph.invoke({"messages": []})
print(f"Messages in final state: {len(result['messages'])}")
result

### Built-in `MessagesState`

LangGraph provides a pre-built `MessagesState` that already includes the `add_messages` reducer. Use it as a shortcut:

In [ ]:
from langgraph.graph import MessagesState
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(model="gpt-4o-mini")

def chat_node(state: MessagesState):
    response = llm.invoke(state["messages"])
    return {"messages": [response]}

workflow = StateGraph(MessagesState)
workflow.add_node("chat", chat_node)
workflow.add_edge(START, "chat")
workflow.add_edge("chat", END)
app = workflow.compile()

display(Image(app.get_graph().draw_mermaid_png()))

In [ ]:
result = app.invoke({"messages": [HumanMessage("Tell me a joke about graphs.")]})
result["messages"][-1].pretty_print()

---
## 3. LLM Workflow Patterns

Now let's explore three essential patterns for building LLM workflows with LangGraph. These patterns are inspired by [Anthropic's article on building effective agents](https://www.anthropic.com/research/building-effective-agents).

<img src="./assets-resources/2025-02-10-20-01-19.png" width=50%>

### Pattern 1: Chaining (Quiz Generation)

A sequential pipeline: **create → review → improve**. The review step uses structured output to decide whether the quiz is good enough.

In [ ]:
from pydantic import BaseModel, Field
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(model="gpt-4o-mini")

# Structured output for the review step
class ReviewOutput(BaseModel):
    quiz_score: str = Field(description="'APPROVED' or 'TO-REVIEW'")

llm_reviewer = llm.with_structured_output(ReviewOutput)

In [ ]:
class QuizState(TypedDict):
    input_source: str
    n_questions: str
    quiz: str
    quiz_quality_score: str
    improved_quiz: str


def create_quiz(state: QuizState) -> QuizState:
    n_questions = state["n_questions"]
    input_source = state["input_source"]
    quiz = llm.invoke(f"Create a markdown styled quiz with {n_questions} questions given this content:\n {input_source}")
    return {"quiz": quiz.content}


def review_quiz(state: QuizState) -> QuizState:
    input_source = state["input_source"]
    initial_quiz = state["quiz"]
    review_prompt = f"""You are a reviewer that scores the quality of quizzes based on input content.
    Source: '''{input_source}'''
    Quiz: '''{initial_quiz}'''
    Return 'APPROVED' if the quiz is relevant and comprehensive, 'TO-REVIEW' otherwise."""
    result = llm_reviewer.invoke(review_prompt)
    return {"quiz_quality_score": result.quiz_score}


def route_quiz_feedback(state: QuizState):
    if state["quiz_quality_score"] == "APPROVED":
        return "approved"
    return "improve"


def write_improved_quiz(state: QuizState) -> QuizState:
    input_source = state["input_source"]
    initial_quiz = state["quiz"]
    prompt = f"""Improve this quiz based on the source material.
    Source: '''{input_source}'''
    Current quiz: '''{initial_quiz}'''
    Output ONLY the improved quiz as a numbered list."""
    improved = llm.invoke(prompt)
    return {"improved_quiz": improved.content}

In [ ]:
workflow = StateGraph(QuizState)

workflow.add_node("create_quiz", create_quiz)
workflow.add_node("review_quiz", review_quiz)
workflow.add_node("improve_quiz", write_improved_quiz)

workflow.add_edge(START, "create_quiz")
workflow.add_edge("create_quiz", "review_quiz")
workflow.add_conditional_edges("review_quiz", route_quiz_feedback, {"approved": END, "improve": "improve_quiz"})
workflow.add_edge("improve_quiz", END)

graph = workflow.compile()
display(Image(graph.get_graph().draw_mermaid_png()))

In [ ]:
from IPython.display import Markdown

input_source = """LangGraph core concepts:
1. Persistence means keeping state information throughout graph execution.
2. Two fundamental questions: which variables to track, and which artifacts are useful for debugging.
3. Sub-graphs are graphs used as nodes in other graphs.
4. Command is a node that updates state and routes to other nodes."""

output = graph.invoke({"input_source": input_source, "n_questions": "3"})
Markdown(output["quiz"])

### Pattern 2: Parallel Execution

Multiple nodes can run in parallel from `START`, then join at a final node. This is useful when you need multiple independent LLM calls that combine into a single output.

In [ ]:
class ExplanationState(TypedDict):
    question: str
    analogy: str
    examples: str
    plain_english: str
    technical_definition: str
    full_explanation: str


def generate_analogy(state: ExplanationState) -> ExplanationState:
    question = state["question"]
    output = llm.invoke(f"Explain this by providing an analogy: {question}")
    return {"analogy": output.content}

def generate_example(state: ExplanationState) -> ExplanationState:
    question = state["question"]
    output = llm.invoke(f"Provide 3 examples to demonstrate: {question}")
    return {"examples": output.content}

def explain_plain_english(state: ExplanationState) -> ExplanationState:
    question = state["question"]
    output = llm.invoke(f"Explain in simple plain english in 2 paragraphs: {question}")
    return {"plain_english": output.content}

def explain_technical(state: ExplanationState) -> ExplanationState:
    question = state["question"]
    output = llm.invoke(f"Give a rigorous technical definition in 2 paragraphs: {question}")
    return {"technical_definition": output.content}

def generate_final_explanation(state: ExplanationState) -> ExplanationState:
    full = f"""# Full Explanation\n\n## Analogy\n{state['analogy']}\n\n## Examples\n{state['examples']}\n\n## Plain English\n{state['plain_english']}\n\n## Technical Definition\n{state['technical_definition']}"""
    return {"full_explanation": full}

In [ ]:
workflow = StateGraph(ExplanationState)

workflow.add_node("generate_analogy", generate_analogy)
workflow.add_node("generate_example", generate_example)
workflow.add_node("explain_plain_english", explain_plain_english)
workflow.add_node("explain_technical", explain_technical)
workflow.add_node("generate_final_explanation", generate_final_explanation)

# All 4 nodes run in parallel from START
workflow.add_edge(START, "generate_analogy")
workflow.add_edge(START, "generate_example")
workflow.add_edge(START, "explain_plain_english")
workflow.add_edge(START, "explain_technical")

# All join at the final node
workflow.add_edge("generate_analogy", "generate_final_explanation")
workflow.add_edge("generate_example", "generate_final_explanation")
workflow.add_edge("explain_plain_english", "generate_final_explanation")
workflow.add_edge("explain_technical", "generate_final_explanation")
workflow.add_edge("generate_final_explanation", END)

graph = workflow.compile()
display(Image(graph.get_graph().draw_mermaid_png()))

In [ ]:
output = graph.invoke({"question": "Explain the concept of self-attention in transformers."})
Markdown(output["full_explanation"])

### Pattern 3: Routing with Structured Output

An LLM classifies the input and routes to a specialized "expert" node. We use Pydantic's structured output to get reliable routing decisions.

In [ ]:
class RoutingState(TypedDict):
    question: str
    category: str | None
    answer: str | None

class QuestionType(BaseModel):
    category: str = Field(description="Category: CODE, MATH, or GENERAL")

llm_classifier = llm.with_structured_output(QuestionType)

# Router: classify the question
def route_question(state: RoutingState):
    response = llm_classifier.invoke(
        f"""Classify this question into exactly ONE category:
        - CODE if about programming/coding
        - MATH if about mathematical calculations
        - GENERAL for general knowledge
        Question: {state['question']}"""
    )
    print(f"Classified as: {response.category}")
    return {"category": response.category}

# Specialized expert nodes
def code_expert(state: RoutingState):
    print("Using code expert!")
    return {"answer": llm.invoke(f"As a coding expert, answer: {state['question']}").content}

def math_expert(state: RoutingState):
    print("Using math expert!")
    return {"answer": llm.invoke(f"As a math expert, solve: {state['question']}").content}

def general_expert(state: RoutingState):
    print("Using general expert!")
    return {"answer": llm.invoke(f"Answer this question: {state['question']}").content}

# Routing logic
def router(state: RoutingState):
    if state["category"] == "CODE":
        return "code_expert"
    elif state["category"] == "MATH":
        return "math_expert"
    return "general_expert"

# Build the graph
workflow = StateGraph(RoutingState)
workflow.add_node("route", route_question)
workflow.add_node("code_expert", code_expert)
workflow.add_node("math_expert", math_expert)
workflow.add_node("general_expert", general_expert)

workflow.add_edge(START, "route")
workflow.add_conditional_edges("route", router, {
    "code_expert": "code_expert",
    "math_expert": "math_expert",
    "general_expert": "general_expert"
})
workflow.add_edge("code_expert", END)
workflow.add_edge("math_expert", END)
workflow.add_edge("general_expert", END)

graph = workflow.compile()
display(Image(graph.get_graph().draw_mermaid_png()))

In [ ]:
graph.invoke({"question": "What is the time complexity of quicksort?", "category": None, "answer": None})

In [ ]:
graph.invoke({"question": "What is the integral of x^2?", "category": None, "answer": None})

---
## 4. Tool Integration

LLMs can't execute code — but they can generate **tool calls** that describe which function to call with what arguments. LangGraph provides `ToolNode` and `tools_condition` to automate this loop.

### Binding Tools to an LLM

When we `bind_tools` to an LLM, it can now generate structured tool call requests (but doesn't execute them).

In [ ]:
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(model="gpt-4o-mini")

def sum_2_numbers(a: int, b: int) -> int:
    """Takes 2 arguments and returns their sum."""
    return a + b

math_tools = [sum_2_numbers]
llm_with_tools = llm.bind_tools(math_tools)

# The LLM generates a tool call, but doesn't execute it
output = llm_with_tools.invoke("What is the sum of 10 + 23?")
print("Content:", output.content)
print("Tool calls:", output.tool_calls)

We could execute the tool call manually:

In [ ]:
tool_map = {'sum_2_numbers': sum_2_numbers}
func = tool_map[output.tool_calls[0]['name']]
a = output.tool_calls[0]['args']['a']
b = output.tool_calls[0]['args']['b']
print(f"Calling {func.__name__}({a}, {b}) = {func(a, b)}")

### `ToolNode` + `tools_condition`: The Agentic Loop

Instead of manually executing tools, LangGraph provides:
- **`ToolNode`** — A node that automatically executes any tool calls from the LLM
- **`tools_condition`** — A routing function that sends to `ToolNode` if the LLM made tool calls, otherwise goes to `END`

Together they create the **agentic loop**: LLM → Tools → LLM → ... → END

In [ ]:
from langgraph.graph import StateGraph, START, END, MessagesState
from langgraph.prebuilt import ToolNode, tools_condition

def sum_2_numbers(a: int, b: int) -> int:
    """Takes 2 arguments and returns their sum."""
    return a + b

tool_node = ToolNode([sum_2_numbers])

def llm_node(state: MessagesState) -> MessagesState:
    messages = state["messages"]
    response = llm_with_tools.invoke(messages)
    return {"messages": [response]}

workflow = StateGraph(MessagesState)
workflow.add_node("llm", llm_node)
workflow.add_node("tools", tool_node)

workflow.add_edge(START, "llm")
workflow.add_conditional_edges("llm", tools_condition)  # Routes to 'tools' or END
workflow.add_edge("tools", "llm")  # Loop back to LLM after tool execution

graph = workflow.compile()
display(Image(graph.get_graph().draw_mermaid_png()))

In [ ]:
result = graph.invoke({"messages": ["Sum of 10 and 4?"]})
for msg in result["messages"]:
    msg.pretty_print()

### Capstone: Chatbot with Web Search

Let's put it all together — a chatbot that can search the web using TavilySearch, built with the same `ToolNode` + `tools_condition` pattern.

In [ ]:
_set_env("TAVILY_API_KEY")

In [ ]:
from langchain_tavily import TavilySearch
from langchain.chat_models import init_chat_model

llm = init_chat_model("openai:gpt-4o-mini")
web_search_tool = TavilySearch(max_results=2)
tools = [web_search_tool]

llm_with_tools = llm.bind_tools(tools)

def chatbot(state: MessagesState):
    return {"messages": [llm_with_tools.invoke(state["messages"])]}

tool_node = ToolNode(tools=tools)

graph_builder = StateGraph(MessagesState)
graph_builder.add_node("chatbot", chatbot)
graph_builder.add_node("tools", tool_node)
graph_builder.add_conditional_edges("chatbot", tools_condition)
graph_builder.add_edge("tools", "chatbot")
graph_builder.add_edge(START, "chatbot")

graph = graph_builder.compile()
display(Image(graph.get_graph().draw_mermaid_png()))

In [ ]:
result = graph.invoke({"messages": ["What are the latest news about LangGraph?"]})
result["messages"][-1].pretty_print()

---
## Summary

In this notebook we covered the core building blocks of LangGraph:

| Concept | Key API |
|---------|--------|
| State definition | `TypedDict` |
| Graph building | `StateGraph`, `START`, `END` |
| Routing | `add_conditional_edges()` |
| Message accumulation | `Annotated[list, add_messages]` or `MessagesState` |
| Structured output | `llm.with_structured_output(PydanticModel)` |
| Tool integration | `bind_tools()`, `ToolNode`, `tools_condition` |
| Workflow patterns | Chaining, Parallel, Routing |

**Next**: In Notebook 2, we'll build a full ReAct agent with memory persistence.